# A3 移植 (5/8)：M4 —— AIV 融合 C8 pack

> **一句话结论**：A3 没有原生 int8 KV cache，原来 6–7 个碎 kernel 组成装链约 +5.9 ms/步；新写一个 AIV 融合算子把整条链压成一次调用，388 → 21.4 µs（18×）。

## 1. 问题：软件 C8 税

每层 decode 都要把 bf16 KV 变成 656B 行。上游碎算子链拆开是：

| 步骤 | 算子 | ms/步 |
| --- | --- | --- |
| 切分 512/64 | Slice ×2 | 0.93 |
| rmsnorm | `npu_rms_norm` | 1.06 |
| RoPE | `npu_interleave_rope` | 0.98 |
| 4×128 块量化 | `npu_dynamic_block_quant` | 1.02 |
| 分量打包 | `pack_c8_components`（torch.cat） | 0.61 |
| 转置/视图辅助 | `TransposeBMM` | 0.99 |
| scatter 到池 | Triton `scatter_packed_kv` | 0.29 |
| **合计** | | **≈5.9** |

## 2. 融合算子 `HisparsePackC8`

一行一核，单算子完成：

1. rmsnorm：fp32 计算，输出舍入到 bf16；
2. 4 个 128 块 int8 量化：absmax/127 作为 fp32 scale；
3. RoPE lane 代数；
4. 656B 组装；
5. scatter 写回池。

输出 rope / latent / scales 三个预分配分量。

## 3. 语义先钉死，再融合

融合前用探针逐字段对齐 vendor 链：rmsnorm 舍入规则、scale codes 取整规则、RoPE 逐位对齐；6 组寻址对抗用例逐字节一致。

## 4. 门控与回退

- 环境变量 `HISPARSE_PACK_FUSED=1` 启用；
- 仅 rows ≤ 64 的 decode 路径；
- prefill、以及 A5 路径回退原碎链。

## 5. 收益

- 算子：388 → 21.4 µs/call（18×）；
- e2e 再省 2.5 ms/步。

## 6. 小结

- M4 = 碎链税 → AIV 一行一核融合；
- 语义先对齐再融合；门控仅限 decode 小批量；
- 下一篇看 A3 改动后的一步前向与时序约束。

**下一篇**：`06-a3-forward-flow.ipynb`
